In [5]:
!pip install gurobipy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ANDRESSA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import os
os.environ['GRB_LICENSE_FILE'] = 'C:\\Users\\ANDRESSA\\Desktop\\ulsr_py\\lic gurobi.lic'

In [7]:
THREADS = 0


In [8]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time


In [9]:
def read_instance_ulsr(datafile):
    with open(datafile, 'r') as file:
        linhas = [a.strip() for a in file.readlines()]

    aux = 0
    N = int(linhas[aux])

    H  = np.zeros(N)
    P  = np.zeros(N)
    F  = np.zeros(N)
    HR = np.zeros(N)
    PR = np.zeros(N)
    FR = np.zeros(N)

    aux += 1
    FR[:] = float(linhas[aux])

    aux += 1
    F[:] = float(linhas[aux])

    aux += 1
    HR[:] = float(linhas[aux])

    aux += 1
    H[:] = float(linhas[aux])

    aux += 1
    D = np.fromstring(linhas[aux], dtype=float, sep=' ')

    aux += 1
    R = np.fromstring(linhas[aux], dtype=float, sep=' ')

    return {
        "N": N,
        "H": H,
        "P": P,
        "F": F,
        "HR": HR,
        "PR": PR,
        "FR": FR,
        "D": D,
        "R": R
    }

In [10]:
def solve_ulsr(inst, time_limit=10800, seed=0):
    N  = inst["N"]
    H  = inst["H"]
    P  = inst["P"]
    F  = inst["F"]
    HR = inst["HR"]
    PR = inst["PR"]
    FR = inst["FR"]
    D  = inst["D"]
    R  = inst["R"]

    m = gp.Model("ulsr")

    m.setParam("OutputFlag", 0)
    m.setParam("TimeLimit", time_limit)
    m.setParam("Seed", seed)
    m.setParam("Threads", THREADS)
    m.setParam("MIPGap", 0.0001)

    x  = m.addVars(N, name="x")
    s  = m.addVars(N, name="s")
    y  = m.addVars(N, vtype=GRB.BINARY, name="y")
    xr = m.addVars(N, name="xr")
    sr = m.addVars(N, name="sr")
    yr = m.addVars(N, vtype=GRB.BINARY, name="yr")

    m.setObjective(
        gp.quicksum(
            P[i]*x[i] + H[i]*s[i] + F[i]*y[i]
            + PR[i]*xr[i] + HR[i]*sr[i] + FR[i]*yr[i]
            for i in range(N)
        ),
        GRB.MINIMIZE
    )

    m.addConstr(x[0] + xr[0] - s[0] == D[0])
    for i in range(1, N):
        m.addConstr(s[i-1] + x[i] + xr[i] - s[i] == D[i])

    m.addConstr(-xr[0] - sr[0] == -R[0])
    for i in range(1, N):
        m.addConstr(sr[i-1] - xr[i] - sr[i] == -R[i])

    for i in range(N):
        m.addConstr(x[i] <= D[i:N].sum() * y[i])

    for i in range(N):
        m.addConstr(xr[i] <= min(D[i:N].sum(), R[0:i+1].sum()) * yr[i])

    m.addConstr(s[N-1] == 0)

    start = time.time()
    m.optimize()
    runtime = time.time() - start

    if m.SolCount > 0:
        objval   = m.ObjVal
        objbound = m.ObjBound
        mipgap   = m.MIPGap
        nodes    = m.NodeCount
        opt      = 1 if mipgap <= 0.0001 else 0
    else:
        objval = objbound = None
        mipgap = 1.0
        nodes  = 0
        opt    = 0

    return objval, objbound, mipgap, runtime, nodes, opt

In [11]:
if __name__ == "__main__":

    instances = "C:\\Users\\ANDRESSA\\Desktop\\ulsr_py\\instances\\sifaleras\\"
    result = "results"

    # cria o diretório de resultados
    if not os.path.exists(result):
        os.makedirs(result)

    # limpa o arquivo antes de escrever
    csv_path = os.path.join(result, "uls_std_mip.csv")
    with open(csv_path, "w") as f:
        f.write("instance;objval;objbound;mipgap;time;nodes;opt\n")

    tab = []

    for dim in [52]:
        for id in [10]:
            datafile = f"{instances}/{dim}_{id}.txt"
            instance_name = os.path.basename(datafile)
            print(f"Resolvendo instancia {instance_name}")

            inst = read_instance_ulsr(datafile)
            objval, objbound, mipgap, runtime, nodes, opt = solve_ulsr(inst)

            # escreve no CSV
            with open(csv_path, "a") as f:
                f.write(
                    instance_name + ";" +
                    str(round(objval, 2)) + ";" +
                    str(round(objbound, 2)) + ";" +
                    str(round(mipgap, 4)) + ";" +
                    str(round(runtime, 2)) + ";" +
                    str(round(nodes, 2)) + ";" +
                    str(opt) + "\n"
                )

            # guarda para o DataFrame
            tab.append({
                "instance": instance_name, # Use instance_name here
                "objval": objval,
                "objbound": objbound,
                "mipgap": mipgap,
                "time": runtime,
                "nodes": nodes,
                "opt": opt
            })

    tab = pd.DataFrame(tab)
    tab

Resolvendo instancia 52_10.txt
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2709503
Academic license 2709503 - for non-commercial use only - registered to an___@alu.ufc.br


In [12]:
import os

resume = pd.DataFrame({
    'instance': "resume",
    'objval': tab["objval"].astype(float).mean(),
    'objbound': tab["objbound"].astype(float).mean(),
    'mipgap': tab['mipgap'].astype(float).mean(),
    'time': tab['time'].astype(float).mean(),
    'nodes': tab['nodes'].astype(float).mean(),
    'opt': tab['opt'].astype(int).sum(),
}, index=["uls_mip"])

tab = pd.concat([tab, resume], ignore_index=True)

# Ensure the 'instance' column contains only the basename for all rows
tab['instance'] = tab['instance'].apply(os.path.basename)

tab["objval"] = tab["objval"].round(2)
tab["objbound"] = tab["objbound"].round(2)
tab["mipgap"] = tab["mipgap"].round(4)
tab["time"] = tab["time"].round(2)
tab["nodes"] = tab["nodes"].round(2)
tab["opt"] = tab["opt"].astype("Int64")

tab

,instance,objval,objbound,mipgap,time,nodes,opt
0,52_10.txt,10812.8,10812.14,0.0001,3953.42,4512344.0,1
1,resume,10812.8,10812.14,0.0001,3953.42,4512344.0,1


In [13]:
!pip install jinja2

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ANDRESSA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
print(
    tab[['instance','objval','objbound','mipgap','time','nodes','opt']].
    to_latex(index=False,float_format="%.2f")
)

\begin{tabular}{lrrrrrr}
\toprule
instance & objval & objbound & mipgap & time & nodes & opt \\
\midrule
52_10.txt & 10812.80 & 10812.14 & 0.00 & 3953.42 & 4512344.00 & 1 \\
resume & 10812.80 & 10812.14 & 0.00 & 3953.42 & 4512344.00 & 1 \\
\bottomrule
\end{tabular}



In [15]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ANDRESSA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [16]:
tab[['instance','objval','objbound','mipgap','time','nodes','opt']] \
    .to_excel("results/results.xlsx", index=False)
